# MLOps in 15 Minutes: From Notebook to Production

**Audience:** beginners · **Length:** about 15–20 minutes · **Project:** `iris-mlops-pipeline`

| # | Section | Time |
|---|---------|------|
| 0 | Setup | 1 min |
| 1 | What is MLOps? | 2 min |
| 2 | Data | 2 min |
| 3 | Train and evaluate with a script | 2 min |
| 4 | The quality gate | 3 min |
| 5 | The model artifact | 2 min |
| 6 | Reproducibility | 2 min |
| 7 | Serving the model | 2 min |
| 8 | Monitoring and drift | 2 min |
| 9 | Recap and Q&A | 2 min |

**How to use this notebook:** run the cells from top to bottom. Along the way you'll find four kinds of notes:

- 💡 **Why this matters**: the key idea behind each step
- 🤔 **Think about it**: questions to reflect on or discuss
- 🛠️ **Try it yourself**: small hands-on exercises
- 📌 **Tip**: practical advice for running the notebook smoothly

> 📌 **Tip:** Restart the kernel and run all cells once to check everything is installed. Then restart again and step through the cells one at a time with Shift+Enter. The whole walkthrough takes about 15–20 minutes.

In [30]:
# ---------------------------------------------------------------
# SETUP — run this cell first. It works locally AND in Google Colab.
# Locally, choose the project's ".venv" kernel (created by `uv sync`).
# ---------------------------------------------------------------
import os          # change folders
import sys         # check where we are running and which Python
import subprocess  # run terminal commands from Python
from pathlib import Path

# 👉 Replace ambi1999 with your GitHub username (only needed for Colab).
GITHUB_REPO = "https://github.com/ambi1999/iris-mlops-pipeline.git"

IN_COLAB = "google.colab" in sys.modules  # True when running in Google Colab

if IN_COLAB:
    # Colab starts with an empty folder, so download the project from GitHub.
    if Path.cwd().name != "iris-mlops-pipeline":  # skip if this cell was already run
        if not Path("iris-mlops-pipeline").exists():
            subprocess.run(["git", "clone", GITHUB_REPO], check=True)
        os.chdir("iris-mlops-pipeline")
    # Colab has its own Python, so we use uv to turn uv.lock into a plain
    # package list and install those exact versions into Colab's Python.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run([sys.executable, "-m", "uv", "export", "--frozen", "--all-groups",
                    "--no-hashes", "--no-emit-project", "-o", "colab-requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "colab-requirements.txt"], check=True)
elif Path.cwd().name == "notebooks":
    # Locally, the notebook starts inside notebooks/. Step up to the project
    # root so that train.py, app.py, etc. can be found.
    os.chdir("..")


def run(command):
    """Run a terminal command and print its output, like typing it in a terminal."""
    print(f"$ {' '.join(command)}\n")
    result = subprocess.run(command, capture_output=True, text=True)
    print(result.stdout + result.stderr)
    print(f"(exit code {result.returncode}: {'success ✅' if result.returncode == 0 else 'failure ❌'})")
    return result


print("Running in:", "Google Colab" if IN_COLAB else "local Jupyter")
print("Project folder:", Path.cwd())

Running in: local Jupyter
Project folder: c:\Ambi\workspace\01MLOpsProject\iris-mlops-pipeline


## 1. What is MLOps? *(≈2 min)*

Training a model in a notebook is only the beginning. **MLOps** (Machine Learning Operations) is the set of practices that take a model from "it works on my laptop" to "it reliably serves real users, and we know when it stops working."

```
  Data ──► Train ──► Evaluate ──► Quality gate ──► Package ──► Serve ──► Monitor
   ▲                                                                       │
   └──────────────────── retrain when the world changes ◄──────────────────┘
```

Three big ideas we'll see today:
- **Automation:** the same steps run the same way every time, without a human clicking buttons.
- **Reproducibility:** anyone, anywhere, can rebuild the exact same model.
- **Monitoring:** models get worse over time, and we need to notice.

> 💡 **Why this matters:** Think of a restaurant. A data scientist with a notebook is like a chef who cooked one great dish at home. MLOps is the restaurant kitchen: written recipes (code), quality checks before plates leave (tests and gates), the same taste every night (reproducibility), and listening to customer feedback (monitoring).

> 🤔 **Think about it:** What could go wrong if you emailed your model file to a colleague and they put it straight on a website? Keep your answers in mind: the rest of this notebook tackles those problems one by one.

## 2. Data *(≈2 min)*

Every model starts with data. We use the classic **Iris dataset**: 150 flowers, 4 measurements each, 3 species. It's tiny, so everything runs in seconds.

In [31]:
# Load the dataset as a table (a pandas DataFrame) so it's easy to look at.
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
df = iris.frame                                   # inputs + target in one table
df["species"] = df["target"].map(dict(enumerate(iris.target_names)))  # 0/1/2 -> names

print("Rows, columns:", df.shape)
print("\nFlowers per species:")
print(df["species"].value_counts().to_string())
df.head()                                         # show the first 5 rows

Rows, columns: (150, 6)

Flowers per species:
species
setosa        50
versicolor    50
virginica     50


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target,species
0,5.1,3.5,1.4,0.2,0,setosa
1,4.9,3.0,1.4,0.2,0,setosa
2,4.7,3.2,1.3,0.2,0,setosa
3,4.6,3.1,1.5,0.2,0,setosa
4,5.0,3.6,1.4,0.2,0,setosa


> 💡 **Why this matters:** The four measurement columns are the *features* (inputs), and the species column is the *label* the model must learn to predict. A model is only as good as its data. In real projects data changes over time (new rows, corrected errors), so teams *version* their data just like code, with tools such as **DVC**. If your data changes silently, your model changes silently too.

## 3. Train and evaluate with a script *(≈2 min)*

Notice we **don't** train the model inside this notebook. We run `train.py`, the **exact same script** that our automated pipeline (CI) runs on GitHub.

Why? Notebooks are great for exploring, but they're easy to run out of order and hard to automate. A script runs the same way every time, top to bottom.

In [32]:
# Run the training script exactly like the CI pipeline does.
run([sys.executable, "train.py"])

# The script also saved its score in metrics.json. Let's read it.
print(Path("metrics.json").read_text())

$ c:\Ambi\workspace\01MLOpsProject\iris-mlops-pipeline\.venv\Scripts\python.exe train.py

Test accuracy: 1.000
Model saved to model.joblib

(exit code 0: success ✅)
{
  "accuracy": 1.0
}


> 💡 **Why this matters:** Two files just appeared: `model.joblib` (the trained model) and `metrics.json` (its score). The score comes from a **train/test split**: 20% of the flowers were hidden from the model during training and used only for testing. It's like giving a student exam questions they haven't seen; scoring them on the practice questions would be cheating.

> 🛠️ **Try it yourself:** Open `train.py` and read through it. Every line is commented, and it's the exact script the CI pipeline runs.

## 4. The quality gate *(≈3 min)*

What if someone changes the code and accidentally makes the model worse? Without a check, the bad model would go straight to users.

A **quality gate** is an automatic rule: *if accuracy is below 85%, stop everything.* `train.py` ends with exactly this check. Let's simulate it with a deliberately "lazy" model that always guesses the same species.

In [33]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

MIN_ACCURACY = 0.85  # the same threshold used in train.py

def quality_gate(name, model):
    """Train a model, score it, and decide whether it is allowed to ship."""
    X_train, X_test, y_train, y_test = train_test_split(
        iris.data, iris.target, test_size=0.2, random_state=42)
    model.fit(X_train, y_train)
    accuracy = accuracy_score(y_test, model.predict(X_test))
    verdict = "PASS ✅ → can be deployed" if accuracy >= MIN_ACCURACY else "BLOCKED ❌ → pipeline stops"
    print(f"{name:<22} accuracy = {accuracy:.2f}   {verdict}")

# A "lazy" model that always predicts the most common species.
quality_gate("Lazy model", DummyClassifier(strategy="most_frequent"))

# Our real model.
quality_gate("Logistic regression", LogisticRegression(max_iter=200))

Lazy model             accuracy = 0.30   BLOCKED ❌ → pipeline stops
Logistic regression    accuracy = 1.00   PASS ✅ → can be deployed


> 💡 **Why this matters:** The lazy model scores about 30%, no better than random guessing between 3 species, so the gate blocks it. In the real pipeline, `train.py` exits with an error, GitHub Actions turns red, and the bad model never reaches users. Nobody has to remember to check; the system does it. That's the heart of MLOps: turning "please remember to check" into "it's checked automatically." Real teams also gate on things like speed, fairness across groups, or "must beat the model already in production."

> 🤔 **Think about it:** What else would you want to check before letting a model ship?

## 5. The model artifact *(≈2 min)*

After training, the model is just a **file**: `model.joblib`. This file is called an **artifact**. It can be saved, copied, versioned, and loaded anywhere, without retraining.

In [34]:
import joblib

size_kb = Path("model.joblib").stat().st_size / 1024
print(f"model.joblib is {size_kb:.1f} KB")

# Load the model from the file, exactly as the web API does at startup.
model = joblib.load("model.joblib")

# Predict one new flower: [sepal length, sepal width, petal length, petal width] in cm.
new_flower = [[6.7, 3.0, 5.2, 2.3]]
predicted = iris.target_names[model.predict(new_flower)[0]]
probabilities = model.predict_proba(new_flower)[0]

print("Prediction:", predicted)
for species, p in zip(iris.target_names, probabilities):
    print(f"  {species:<12} {p:.0%}")

model.joblib is 1.0 KB
Prediction: virginica
  setosa       0%
  versicolor   10%
  virginica    90%


> 💡 **Why this matters:** The model also reports how *confident* it is, not just its answer. In the GitHub pipeline, this file is uploaded after every run (look for **Artifacts** on the run's page), so every version is kept. Larger teams use a **model registry** such as MLflow, which labels versions as "staging" or "production" and makes rolling back to an older model a one-click job.

## 6. Reproducibility *(≈2 min)*

If you can't rebuild a model, you can't debug it, audit it, or trust it. Two things in this project make training reproducible:
1. A fixed **random seed** (`random_state=42`), so the data is shuffled the same way every time.
2. **Locked package versions**: `uv.lock` records the exact version of every package, and `.python-version` fixes the Python version (3.13).

In [35]:
def train_and_score(seed):
    X_train, X_test, y_train, y_test = train_test_split(
        iris.data, iris.target, test_size=0.2, random_state=seed)
    m = LogisticRegression(max_iter=200).fit(X_train, y_train)
    return accuracy_score(y_test, m.predict(X_test))

print("Same seed, run 1:", f"{train_and_score(42):.3f}")
print("Same seed, run 2:", f"{train_and_score(42):.3f}   ← identical")
print()
for seed in [1, 7, 99]:
    print(f"Different seed {seed:>2}:", f"{train_and_score(seed):.3f}")

Same seed, run 1: 1.000
Same seed, run 2: 1.000   ← identical

Different seed  1: 0.967
Different seed  7: 0.867
Different seed 99: 0.933


> 💡 **Why this matters:** The same seed gives exactly the same result, while different seeds give different scores, even with identical code and data. Anyone can recreate this project's exact environment with one command, `uv sync`. The Dockerfile goes further and fixes the operating system too.

> 🤔 **Think about it:** A colleague reports 93% accuracy and you get 100% with the "same" code. Who's right? Without fixed seeds and locked versions, there's no way to tell.

## 7. Serving the model *(≈2 min)*

Users don't open notebooks; apps talk to the model over the web. `app.py` wraps the model in a small **web API** with two endpoints:
- `GET /health`: "are you alive?"
- `POST /predict`: send measurements, receive a species

Below we call the API in memory with FastAPI's `TestClient`, so no server is needed.

In [36]:
from fastapi.testclient import TestClient
from app import app

client = TestClient(app)

flower = {"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}
response = client.post("/predict", json=flower)
print("Status code:", response.status_code)
print("Response:   ", response.json())

# Invalid input is rejected cleanly instead of crashing the service.
bad = client.post("/predict", json={"sepal_length": "not a number"})
print("\nBad input status code:", bad.status_code, "(422 = rejected input)")

Status code: 200
Response:    {'species': 'setosa'}

Bad input status code: 422 (422 = rejected input)


> 💡 **Why this matters:** This JSON request and response is exactly how a mobile app or website would use the model. The packaging side, running the API inside Docker, is covered in the DevOps notebook.

> 🛠️ **Try it yourself:** In a terminal, run `uv run uvicorn app:app --reload`, then open `http://localhost:8000/docs`. You get a clickable page where you can send your own flower measurements to the model. Press Ctrl+C in the terminal to stop the server.

## 8. Monitoring and drift *(≈2 min)*

A model that's 100% accurate today can quietly get worse. The world changes: new customers, new sensors, new seasons. When incoming data stops looking like the training data, that's called **data drift**.

A simple check compares the average of incoming data to the average of the training data.

In [37]:
import numpy as np

training_means = iris.data.mean()  # average of each measurement in the training data

def drift_check(incoming, threshold=0.25):
    """Flag any feature whose average moved more than 25% from training."""
    change = (incoming.mean() - training_means).abs() / training_means
    for feature, c in change.items():
        status = "DRIFT ⚠️" if c > threshold else "ok"
        print(f"  {feature:<20} changed {c:>5.0%}   {status}")

print("Week 1: incoming flowers look like the training data")
drift_check(iris.data.sample(30, random_state=1))

print("\nWeek 8: a new supplier measures petals in a different way")
shifted = iris.data.sample(30, random_state=2).copy()
shifted["petal length (cm)"] *= 1.6
shifted["petal width (cm)"] *= 1.6
drift_check(shifted)

Week 1: incoming flowers look like the training data
  sepal length (cm)    changed    0%   ok
  sepal width (cm)     changed    5%   ok
  petal length (cm)    changed    6%   ok
  petal width (cm)     changed    7%   ok

Week 8: a new supplier measures petals in a different way
  sepal length (cm)    changed    4%   ok
  sepal width (cm)     changed    0%   ok
  petal length (cm)    changed   42%   DRIFT ⚠️
  petal width (cm)     changed   36%   DRIFT ⚠️


> 💡 **Why this matters:** The model would still happily make predictions on the week-8 data. It won't throw an error; it will just be quietly wrong. That's why monitoring matters. Real systems log every prediction and run checks like this automatically, alerting the team or triggering retraining. That closes the loop in the diagram from section 1.

## 9. Recap *(≈2 min)*

| MLOps idea | Where it lives in this repo |
|------------|-----------------------------|
| Training as a repeatable script | `train.py` |
| Quality gate | end of `train.py` (`MIN_ACCURACY`) |
| Model artifact | `model.joblib`, uploaded by CI |
| Metrics tracking | `metrics.json` |
| Reproducibility | `random_state`, `uv.lock`, `.python-version`, `Dockerfile` |
| Serving | `app.py` (FastAPI) |
| Automation | `.github/workflows/ci.yml` |
| Monitoring | concept shown in section 8 (next step for the repo) |

🤔 **Think about it:**
- Which of these steps would have saved you trouble on a past project?
- What would you add to the quality gate for *your* models?

**Where to go next:** experiment tracking with **MLflow**, data versioning with **DVC**, and deploying the Docker image to a cloud service.

**Next:** continue with the DevOps notebook, `02_devops_demo.ipynb`. MLOps borrows almost all of its ideas from DevOps, and that notebook shows where they come from.